# FIN112 project: can AI beat "tomorrow = today"?

Unit 6, section 6.3 (Hands-on AI-based investment modeling)

You are a junior quant at an investment firm. Your manager wants to know if a machine learning model can predict the next day's closing price of a stock better than a lazy rule: "tomorrow's price will be the same as today's."

The textbook practical builds a Linear Regression and a Random Forest. This project does that, then pushes further. You will test your models against a naive baseline, change the prediction target, and run a small backtest with trading costs. Some results will surprise you. Write down what you see, because the report you submit is about what you found, not about getting a high score.

How to work through this notebook:
- Run the cells top to bottom. Read the markdown before each code cell.
- Cells marked YOUR TURN are for you to write or change.
- Keep a short log of results as you go. You will need it for the report.

## Step 0. Setup

Run this cell once. It installs yfinance (Colab doesn't always have it) and loads the libraries.

In [ ]:
!pip -q install yfinance

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style("whitegrid")
print("Libraries loaded")

## Step 1. Get the data

Set the ticker and dates in the first cell below. The textbook uses Apple (AAPL) from June 2019 to June 2024. Keep that for your first run so your numbers match your classmates'.

If Yahoo Finance is slow or blocks the request, the loader falls back to simulated prices so the notebook still runs. It tells you when that happens. Results on simulated data mean nothing about the market, so don't put them in your report.

In [ ]:
TICKER = "AAPL"
START = "2019-06-01"
END = "2024-06-01"

In [ ]:
def load_prices(ticker, start, end):
    try:
        raw = yf.download(ticker, start=start, end=end, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        if raw.empty:
            raise ValueError("empty download")
        return raw[["Close", "Volume"]].copy(), "yfinance"
    except Exception as e:
        print("Download failed:", e)
        print("Using SIMULATED prices. Do not use these results in your report.")
        rng = np.random.default_rng(42)
        dates = pd.bdate_range(start, end)
        rets = rng.normal(0.0007, 0.018, len(dates))
        close = 150 * np.exp(np.cumsum(rets))
        vol = rng.lognormal(17.5, 0.3, len(dates))
        return pd.DataFrame({"Close": close, "Volume": vol}, index=dates), "simulated"

df, source = load_prices(TICKER, START, END)
print("Data source:", source)
print(df.shape)
df.head()

## Step 2. Look before you model

Always plot the data first. Two things to notice in the chart: the price trends upward over long stretches, and the spread of values is wide. Both will matter later.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df["Close"])
plt.title(f"{TICKER} close price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

print(df["Close"].describe())

plt.figure(figsize=(10, 4))
sns.histplot(df["Volume"], kde=True)
plt.title(f"{TICKER} volume distribution")
plt.show()

Question for your log: the price in the last year is higher than anything in the first two years. Why might that be a problem for a model that learns from the first 80% of the data and is tested on the last 20%?

## Step 3. Build features

Each row is one trading day. The features describe what we know at the close of that day, and the target is the next day's close.

- Prev_Close: yesterday's close
- Daily_Return: today's percentage change
- SMA_5 and SMA_20: 5-day and 20-day simple moving averages
- Volume_Change: percentage change in volume
- Next_Close: tomorrow's close (the target)

The last row has no tomorrow, and the first 20 rows have no 20-day average, so those rows are dropped.

In [ ]:
data = df.copy()
data["Prev_Close"] = data["Close"].shift(1)
data["Daily_Return"] = data["Close"].pct_change()
data["SMA_5"] = data["Close"].rolling(5).mean()
data["SMA_20"] = data["Close"].rolling(20).mean()
data["Volume_Change"] = data["Volume"].pct_change()
data["Next_Close"] = data["Close"].shift(-1)
data = data.dropna()

features = ["Prev_Close", "Daily_Return", "SMA_5", "SMA_20", "Volume", "Volume_Change"]
X = data[features]
y = data["Next_Close"]
print(X.shape, y.shape)
data.head()

## Step 4. Split by time, never at random

Train on the first 80% of days and test on the last 20%. A random split would let the model see days from the future while it learns, and the test score would look better than it deserves.

In [ ]:
split = int(len(data) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print("Train:", X_train.index.min().date(), "to", X_train.index.max().date(), f"({len(X_train)} days)")
print("Test: ", X_test.index.min().date(), "to", X_test.index.max().date(), f"({len(X_test)} days)")
print("Highest price seen in training:", round(data['Close'].iloc[:split].max(), 2))
print("Highest price in test period:  ", round(data['Close'].iloc[split:].max(), 2))

Fit the scaler on the training data only, then apply it to the test data. If you fit it on everything, information from the test period leaks into training.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## Step 5. Train the textbook models and a baseline

We train Linear Regression and Random Forest as in the textbook. We also add a baseline that needs no learning at all: predict that tomorrow's close equals today's close. A model is only useful if it beats this.

In [ ]:
lr = LinearRegression().fit(X_train_s, y_train)
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train_s, y_train)

today_close = data["Close"].iloc[split:]          # the close on the day we predict from
pred_naive = today_close.values
pred_lr = lr.predict(X_test_s)
pred_rf = rf.predict(X_test_s)
print("Models trained")

## Step 6. Score the models

Besides MSE and R-squared from the textbook, we add three more:
- RMSE puts the error back in dollars.
- MAE is the average miss in dollars.
- Directional accuracy is the share of days where the model got the direction right (up or down compared with today). A coin flip scores about 50%.

The naive baseline has no direction, since it always says "no change," so its directional accuracy is shown as not applicable.

In [ ]:
def score(name, actual, pred, today):
    mse = mean_squared_error(actual, pred)
    out = {
        "Model": name,
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(actual, pred),
        "R2": r2_score(actual, pred),
    }
    if name.startswith("Naive"):
        out["Direction %"] = np.nan
    else:
        out["Direction %"] = 100 * np.mean(np.sign(pred - today) == np.sign(actual.values - today))
    return out

results = pd.DataFrame([
    score("Naive (tomorrow = today)", y_test, pred_naive, today_close.values),
    score("Linear Regression", y_test, pred_lr, today_close.values),
    score("Random Forest", y_test, pred_rf, today_close.values),
]).set_index("Model").round(4)
results

Read the table slowly.

R-squared on a price level is nearly always very high, even for the naive baseline, because prices today and prices tomorrow are close. A high R-squared here is not evidence of skill. Compare the RMSE and MAE of each model with the naive row instead. If a model doesn't beat the baseline, it has added nothing.

Log your three RMSE values and the directional accuracy numbers now.

## Step 7. Look at the predictions

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(y_test.index, y_test, label="Actual next close", color="black", linewidth=1.5)
ax.plot(y_test.index, pred_lr, label="Linear Regression", color="red", alpha=0.7)
ax.plot(y_test.index, pred_rf, label="Random Forest", color="green", alpha=0.7)
ax.set_title(f"{TICKER}: actual vs predicted next close (test period)")
ax.set_ylabel("Price")
ax.legend()
plt.show()

Check the Random Forest line when the actual price climbs above the highest price in the training data (you printed both numbers in Step 4). A decision tree can only output values it has seen in training, so a Random Forest cannot predict a price higher than anything it was trained on. Linear Regression can extrapolate. This is one reason the textbook's expectation that Random Forest "generally performs better" doesn't hold up for trending prices.

Write one sentence in your log describing what you see.

## Step 8. Change the target: predict the return, not the price

Prices trend, so a model fed price levels is partly just learning "prices are high now." Returns (the percentage change) are much more stable over time. We rebuild the problem:

- Target: tomorrow's return, Close(t+1) / Close(t) minus 1
- Features: scale-free numbers only (returns, price relative to its average, recent volatility, volume change)

Then we convert the predicted return back into a predicted price and score it the same way.

In [ ]:
r = df.copy()
r["ret_1"] = r["Close"].pct_change()
r["ret_5"] = r["Close"].pct_change(5)
r["dist_sma20"] = r["Close"] / r["Close"].rolling(20).mean() - 1
r["vol_20"] = r["ret_1"].rolling(20).std()
r["vol_chg"] = r["Volume"].pct_change()
r["target_ret"] = r["Close"].shift(-1) / r["Close"] - 1
r = r.dropna()

feat_r = ["ret_1", "ret_5", "dist_sma20", "vol_20", "vol_chg"]
s = int(len(r) * 0.8)
Xr_train, Xr_test = r[feat_r].iloc[:s], r[feat_r].iloc[s:]
yr_train, yr_test = r["target_ret"].iloc[:s], r["target_ret"].iloc[s:]

sc_r = StandardScaler()
Xr_train_s = sc_r.fit_transform(Xr_train)
Xr_test_s = sc_r.transform(Xr_test)

lr_r = LinearRegression().fit(Xr_train_s, yr_train)
rf_r = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1).fit(Xr_train_s, yr_train)

pr_lr = lr_r.predict(Xr_test_s)
pr_rf = rf_r.predict(Xr_test_s)

today_r = r["Close"].iloc[s:]
actual_next = today_r * (1 + yr_test)
price_lr = today_r * (1 + pr_lr)
price_rf = today_r * (1 + pr_rf)

results_r = pd.DataFrame([
    score("Naive (tomorrow = today)", actual_next, today_r.values, today_r.values),
    score("LR on returns", actual_next, price_lr.values, today_r.values),
    score("RF on returns", actual_next, price_rf.values, today_r.values),
]).set_index("Model").round(4)
results_r

Now compare with the Step 6 table. Two things to expect: the price-level error should look similar to the naive baseline (daily moves are small compared with the price), and the directional accuracy will sit close to 50%, maybe a few points either side. That is what a hard prediction problem looks like. Markets are noisy, and a small edge is all anyone gets.

## Step 9. Would it make money? A small backtest

A model can have decent error numbers and still lose money. Here is a simple rule: if the model predicts a positive return for tomorrow, hold the stock for tomorrow, otherwise stay in cash. We charge a trading cost every time the position changes.

This ignores slippage, taxes and position sizing, so treat it as a classroom exercise, not a trading plan.

In [ ]:
COST = 0.001   # 0.1% per change of position. Try 0, 0.001 and 0.003.

def backtest(pred_ret, actual_ret, cost):
    signal = (pred_ret > 0).astype(int)
    trades = signal.diff().abs().fillna(signal.abs())
    strat = signal * actual_ret - trades * cost
    return strat, signal

actual_ret = yr_test.values
idx = yr_test.index

strat_lr, sig_lr = backtest(pd.Series(pr_lr, index=idx), pd.Series(actual_ret, index=idx), COST)
strat_rf, sig_rf = backtest(pd.Series(pr_rf, index=idx), pd.Series(actual_ret, index=idx), COST)
buy_hold = pd.Series(actual_ret, index=idx)

curves = pd.DataFrame({
    "Buy and hold": (1 + buy_hold).cumprod(),
    "LR strategy": (1 + strat_lr).cumprod(),
    "RF strategy": (1 + strat_rf).cumprod(),
})
curves.plot(figsize=(12, 5), title=f"Growth of 1 rupee/dollar in the test period (cost {COST*100:.1f}% per trade)")
plt.show()

summary = pd.DataFrame({
    "Final value": curves.iloc[-1],
    "Trades": [0, int(sig_lr.diff().abs().sum()), int(sig_rf.diff().abs().sum())],
}).round(3)
summary

Questions for your log:
1. Did either strategy beat buy and hold?
2. Set COST to 0 and run again. How much of the result was eaten by costs?
3. If the stock rose strongly in your test period, what does that do to the "stay in cash" days?

## YOUR TURN

Do Task A and Task B. Pick one of C, D or E. Add code cells under each task.

Task A. Change the stock.
Set TICKER to an Indian stock such as "RELIANCE.NS" or "TCS.NS" (Yahoo uses the .NS suffix for NSE). Rerun the notebook from Step 1. Does the pattern from AAPL hold? Report the Step 6 and Step 8 tables for your stock.

Task B. Add two features of your own.
Examples: RSI (14 days), a 5-day volatility measure, the previous day's volume change, the day of the week. Add them to the Step 8 return model. Did the directional accuracy move? Be honest if it didn't.

Task C. Better validation.
Replace the single 80/20 split with sklearn's TimeSeriesSplit (5 folds) and report the average directional accuracy across folds. A single split can flatter a model by luck.

Task D. Tune the Random Forest.
Try max_depth values of 3, 5 and 10 and min_samples_leaf values of 5, 20 and 50 on the return model. Keep the test set untouched while you tune: choose parameters using only the training period.

Task E. Add sentiment (link to section 6.1).
Download 20 or more recent headlines for your stock, score each as positive, negative or neutral (by hand or with a library such as VADER or FinBERT), and compare the average score for each week with that week's return. Report the correlation and say how much data you had. With so few points, the honest conclusion is usually "not enough to tell."

In [ ]:
# Task A and B: write your code here

## Reflection questions for your report

1. Which model had the lowest error on the price-level problem, and did it beat the naive baseline?
2. Why is R-squared a poor way to judge a model that predicts prices?
3. What did the Random Forest do when prices went above anything in the training data?
4. After switching to returns, what was the best directional accuracy you got? Is that difference from 50% big enough to trust, given about 250 test days?
5. Would you put money behind your strategy after the backtest? Give two reasons for or against.
6. Name one ethical or practical risk of deploying a model like this at an investment firm (use the Task 4.3 list in section 6.3 if you need ideas).